# Домашнее задание №4: исследование линейной регрессии

In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('ToyotaCorolla.csv')

print(f"Размер датасета:{df.shape}")

print("\n--- Общая информация ---")
df.info()

print("--- Пропуски по колонкам ---")
print(df.isnull().sum())


numeric_df = df.select_dtypes(include=[np.number])
numeric_df = numeric_df.loc[:,numeric_df.nunique()>1]
corr_matrix = numeric_df.corr()
det = np.linalg.det(corr_matrix)
print(f"Определитель матрицы:{det}")



Размер датасета:(1436, 39)

--- Общая информация ---
<class 'pandas.DataFrame'>
RangeIndex: 1436 entries, 0 to 1435
Data columns (total 39 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   Id                 1436 non-null   int64
 1   Model              1436 non-null   str  
 2   Price              1436 non-null   int64
 3   Age_08_04          1436 non-null   int64
 4   Mfg_Month          1436 non-null   int64
 5   Mfg_Year           1436 non-null   int64
 6   KM                 1436 non-null   int64
 7   Fuel_Type          1436 non-null   str  
 8   HP                 1436 non-null   int64
 9   Met_Color          1436 non-null   int64
 10  Color              1436 non-null   str  
 11  Automatic          1436 non-null   int64
 12  CC                 1436 non-null   int64
 13  Doors              1436 non-null   int64
 14  Cylinders          1436 non-null   int64
 15  Gears              1436 non-null   int64
 16  Quarterly_Tax     

У данных я проверил количесвто пропусков и проверил на мультиколлинеарность посчитав определитель. Определитель очень маленький, если например посмотреть на колонки, то есть Age_08_04, Mfg_Month и Mfg_Year что по сути одно и тоже, также колонка Cylinders которая константа, также Radio и Radio_cassette,то же самое с Airco и Automatic_airco.

In [11]:
columns = ['Id','Mfg_Month','Mfd_Month','Mfg_Year','Radio','Airco']
df = df.drop(columns = columns,errors='ignore')
numeric_df = df.select_dtypes(include=[np.number])
numeric_df = numeric_df.loc[:,numeric_df.nunique()>1]
corr_matrix = numeric_df.corr()
det = np.linalg.det(corr_matrix)
print(f"Определитель матрицы:{det}")



Определитель матрицы:6.166543050494717e-06

----------------
<class 'pandas.DataFrame'>
RangeIndex: 1436 entries, 0 to 1435
Data columns (total 34 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   Model              1436 non-null   str  
 1   Price              1436 non-null   int64
 2   Age_08_04          1436 non-null   int64
 3   KM                 1436 non-null   int64
 4   Fuel_Type          1436 non-null   str  
 5   HP                 1436 non-null   int64
 6   Met_Color          1436 non-null   int64
 7   Color              1436 non-null   str  
 8   Automatic          1436 non-null   int64
 9   CC                 1436 non-null   int64
 10  Doors              1436 non-null   int64
 11  Cylinders          1436 non-null   int64
 12  Gears              1436 non-null   int64
 13  Quarterly_Tax      1436 non-null   int64
 14  Weight             1436 non-null   int64
 15  Mfr_Guarantee      1436 non-null   int64
 16  BOVAG_Guar

# Разделение выборки

In [15]:
from sklearn.model_selection import train_test_split

X = df.select_dtypes(include=[np.number])
X = X.drop('Price',axis=1)
y = df['Price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42)

**Зачем вообще разделять данные** это используется для защиты от переобучения и чтобы проверить его предсказательную способность, если например точность на тренировочных данных 99% а на тестовых 40% то модель переобучилась.

# Обучение моделей

In [16]:
from sklearn.linear_model import LinearRegression,Ridge,Lasso

model = [LinearRegression(),Ridge(),Lasso()]
results = []
for md in model:
    md.fit(X_train,y_train)
    results.append({md:md.score(X_test,y_test)})
print(results)


[{LinearRegression(): 0.8930908864041048}, {Ridge(): 0.893036884325302}, {Lasso(): 0.8935849168054502}]


# Оценка качества и сравнение моделей
**Какую метрику я использовал** я использовал MSE, так как она чувствительна к выбросам, ликвидирует знак если промах по цене в другую сторону и также если предположить что ошибка распределена нормально то уменьшение MSE приводит к наиболее вероятным значениям по методу максимального правдоподобия.
**На какой части выборки вы считали метрики?** считал и там и там, результат показан для расчета для тестовой части выборки.
**Какая модель по итогу справилась лучше?** они все справились примерно одинаково, L1 регуляризация оказалась лучше в 4 знаке.
**Насколько хорошие получились результаты?** Достаточно высокая точность.
**Чем докажете, что ваша модель не переобучилась?** Для этого нужно сравнить метрики для трейна и теста.